In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from peft import get_peft_model, LoraConfig, TaskType

sys.path.append(os.path.abspath('..'))
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS  = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP    = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
TRAIN_PATH   = '../data/raw/train.csv'
MODEL_NAME   = 'bert-base-uncased'
DEVICE       = 'mps' if torch.backends.mps.is_available() else 'cpu'

df = pd.read_csv(TRAIN_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Shape  : {df.shape}')
print(f'Device : {DEVICE}')

In [2]:
df['label'] = df['answer'].map(LABEL_MAP)

q1_answer = int(df.iloc[150]['label'])
print(f'Row 150 answer : {df.iloc[150]["answer"]}')
print(f'Label : {q1_answer}')

Row 150 answer : C
Label : 2


In [3]:
row0         = df.iloc[0]
formatted_b  = str(row0['prompt']) + ' [SEP] ' + str(row0['B'])
q2_answer    = len(formatted_b)

print(f'Formatted string: {formatted_b[:100]}...')
print(f'(char length): {q2_answer}')

Formatted string: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...
(char length): 407


In [4]:
def format_choices(row, max_length=128):
    inputs = [
        str(row['prompt']) + ' [SEP] ' + str(row[col])
        for col in OPTION_COLS
    ]
    encoded = tokenizer(
        inputs,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    # Reshape to [1, num_choices, seq_len]
    input_ids      = encoded['input_ids'].unsqueeze(0)
    attention_mask = encoded['attention_mask'].unsqueeze(0)
    return input_ids, attention_mask

input_ids_row0, attn_mask_row0 = format_choices(row0)

q3_answer = input_ids_row0.shape[1]   # second dimension = num_choices
print(f'input_ids shape : {tuple(input_ids_row0.shape)}')
print(f'(second dimension): {q3_answer}')

input_ids shape : (1, 5, 128)
(second dimension): 5


In [5]:
batch_input_ids = []
for i in range(16):
    ids, _ = format_choices(df.iloc[i])
    batch_input_ids.append(ids)

batch_tensor = torch.cat(batch_input_ids, dim=0)   # [16, 5, 128]
q4_answer    = batch_tensor.numel()                 # total elements

print(f'Batch tensor shape      : {tuple(batch_tensor.shape)}')
print(f'(total token positions): {q4_answer}')

Batch tensor shape      : (16, 5, 128)
(total token positions): 10240


In [6]:
mc_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
mc_model.eval()

with torch.no_grad():
    outputs = mc_model(
        input_ids=input_ids_row0,
        attention_mask=attn_mask_row0
    )

logits    = outputs.logits
q5_answer = logits.shape[1]   # number of logits = number of choices

print(f'Logits shape    : {tuple(logits.shape)}')
print(f'Logits values   : {logits}')
print(f'(number of logits): {q5_answer}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7991.37it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Logits shape    : (1, 5)
Logits values   : tensor([[-0.6932, -0.6651, -0.6784, -0.6662, -0.6831]])
(number of logits): 5


In [7]:
label_row0 = torch.tensor([int(df.iloc[0]['label'])])

with torch.no_grad():
    outputs_with_loss = mc_model(
        input_ids=input_ids_row0,
        attention_mask=attn_mask_row0,
        labels=label_row0
    )

loss      = outputs_with_loss.loss
q6_answer = loss.dim()

print(f'Loss value      : {loss.item():.4f}')
print(f'Loss shape      : {loss.shape}')
print(f'(number of dimensions): {q6_answer}')

Loss value      : 1.5974
Loss shape      : torch.Size([])
(number of dimensions): 0


In [8]:
base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

q7_answer = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f'(trainable parameters): {q7_answer}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23817.67it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Con

trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
(trainable parameters): 295681


In [9]:
def tokenize_row(row, max_length=128):
    inputs = [
        str(row['prompt']) + ' [SEP] ' + str(row[col])
        for col in OPTION_COLS
    ]
    encoded = tokenizer(
        inputs,
        padding='max_length',
        truncation=True,
        max_length=max_length,
    )
    return {
        'input_ids'      : encoded['input_ids'],       # [5, 128]
        'attention_mask' : encoded['attention_mask'],  # [5, 128]
        'labels'         : LABEL_MAP[row['answer']]
    }

first_100    = df.iloc[:100].reset_index(drop=True)
records      = [tokenize_row(first_100.iloc[i]) for i in range(len(first_100))]
hf_dataset   = Dataset.from_list(records)

q8_answer    = len(hf_dataset[0]['input_ids'])   # number of choices
print(f'Dataset size              : {len(hf_dataset)}')
print(f'input_ids shape (item 0)  : [{len(hf_dataset[0]["input_ids"])}, {len(hf_dataset[0]["input_ids"][0])}]')
print(f'(tokenized choices): {q8_answer}')

Dataset size              : 100
input_ids shape (item 0)  : [5, 128]
(tokenized choices): 5


In [ ]:
import torch
from torch.utils.data import default_collate

# Build train dataset — first 32 rows, max_length=64
def tokenize_row_64(row):
    inputs = [
        str(row['prompt']) + ' [SEP] ' + str(row[col])
        for col in OPTION_COLS
    ]
    encoded = tokenizer(
        inputs,
        padding='max_length',
        truncation=True,
        max_length=64,
    )
    return {
        'input_ids'      : encoded['input_ids'],
        'attention_mask' : encoded['attention_mask'],
        'labels'         : LABEL_MAP[row['answer']]
    }

first_32   = df.iloc[:32].reset_index(drop=True)
records_32 = [tokenize_row_64(first_32.iloc[i]) for i in range(32)]
train_ds   = Dataset.from_list(records_32)
train_ds.set_format(type='torch')

# Fresh LoRA model for training
base_for_train = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_train_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS
)
lora_train_model = get_peft_model(base_for_train, lora_train_cfg)

training_args = TrainingArguments(
    output_dir='../models/lora-mcq-tiny',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_steps=999,
    report_to='none',
    no_cuda=True,           # MPS handled via torch; disable CUDA flag
    use_mps_device=True if torch.backends.mps.is_available() else False,
)

trainer = Trainer(
    model=lora_train_model,
    args=training_args,
    train_dataset=train_ds,
)

train_result = trainer.train()
q9_answer    = trainer.state.global_step

print(f'Training metrics : {train_result.metrics}')
print(f'(global_step): {q9_answer}')

In [11]:
lora_train_model.eval()

# Tokenize row index 0 with max_length=64 (same as fine-tuning)
inputs_q10 = [
    str(row0['prompt']) + ' [SEP] ' + str(row0[col])
    for col in OPTION_COLS
]
enc_q10 = tokenizer(
    inputs_q10,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt'
)
input_ids_q10 = enc_q10['input_ids'].unsqueeze(0)
attn_mask_q10 = enc_q10['attention_mask'].unsqueeze(0)

with torch.no_grad():
    out_q10 = lora_train_model(
        input_ids=input_ids_q10,
        attention_mask=attn_mask_q10
    )

probs     = torch.softmax(out_q10.logits, dim=-1)[0]
q10_answer = round(probs[4].item(), 4)   # index 4 = option E

print(f'Logits : {out_q10.logits}')
print(f'Probs  : A={probs[0]:.4f} B={probs[1]:.4f} C={probs[2]:.4f} D={probs[3]:.4f} E={probs[4]:.4f}')
print(f'(prob of option E): {q10_answer}')

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Logits : tensor([[-0.1742, -0.1010, -0.1474, -0.1663, -0.1812]])
Probs  : A=0.1959 B=0.2108 C=0.2012 D=0.1975 E=0.1946
(prob of option E): 0.1946
